## 0. Load datasets

In [38]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
import seaborn as sb

# General utils
from tqdm import tqdm
from os import path

from measurements import get_prev_or_next_frame
from sklearn import preprocessing, model_selection, feature_selection, metrics, inspection, pipeline
from sklearn import ensemble, linear_model

def get_balanced_df_by_category(df,category):
    # assert(logical.dtype == bool)
    
    # Find the category with least members
    categories = {cat:mem for cat, mem in df.groupby(category)}
    num_per_category = np.array([len(mem) for mem in categories.values()])
    smallest_category = list(categories.keys())[num_per_category.argmin()]
    
    output = []
    for cat, member in categories.items():
        if cat == smallest_category:
            output.append(member)
        else:
            output.append(member.sample(num_per_category.min()))
    output = pd.concat(output,ignore_index=True)

    return output
    
def predict_feature(df:pd.DataFrame,
                    classifier,
                    feature2predict,
                    Niter:int=100,
                    rebalance:bool=True,
                    subsample=None,
                    measurement_level_name:str='Measurement'):
    if subsample == None:
        subsample = len(df)
    Y_test = []
    Y_pred = []
    models = []
    for i in tqdm(range(Niter)):

        # rng = np.random.RandomState(0)
        df_ = get_balanced_df_by_category(df, df[feature2predict,'Meta'].values)
        
        pipe = pipeline.make_pipeline(preprocessing.StandardScaler(),
                                      classifier() )
        
        y = df_[feature2predict,'Meta'].astype(float)
        X = df_.xs(measurement_level_name,level=1,axis=1).astype(float)
        X[np.isinf(X)] = np.nan
        X_train,X_test,y_train,y_test = model_selection.train_test_split(X,y)
        
        model = pipe.fit(X_train,y_train)
        models.append(model)
        
        Y_test.append(y_test)
        Y_pred.append(model.predict(X_test))
        
    return [np.stack((np.array(Y_test),np.array(Y_pred))),models]
    

In [15]:

dataset_dir = '/Users/xies/Library/CloudStorage/OneDrive-Stanford/Skin/Mesa et al/Lineage models/Dataset pickles'
all_df = pd.read_pickle(path.join(dataset_dir,f'all_df.pkl'))
divisions = pd.read_pickle(path.join(dataset_dir,f'divisions.pkl'))

# censor the 'answers'
features2drop = [(f,a) for f,a in divisions.columns if 'Time' in f]
features2drop += [(f,a) for f,a in divisions.columns if 'smoothed' in f and not 'rate' in f]
features2drop += [(f,a) for f,a in divisions.columns if 'standard' in f and not 'rate' in f]
features2drop += [(f,a) for f,a in divisions.columns if 'cell coords' in f and not 'rate' in f]
features2drop += ['Z']
divisions = divisions.drop(columns=features2drop)
print(len(divisions))


668


/var/folders/zp/5k1kvm9j0t19cvq8h628kr8h0000gn/T/ipykernel_67612/4286720115.py:11: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  divisions = divisions.drop(columns=features2drop)


In [33]:
Xcols = divisions.columns[divisions.columns.get_level_values(1) != 'Meta']
print(Xcols)

MultiIndex([(                                           'Nuclear volume', ...),
            (                                         'Nuclear solidity', ...),
            (                                           'Nuclear height', ...),
            (                                              'Cell volume', ...),
            (                                          'Axial component', ...),
            (                                       'Planar component 1', ...),
            (                                       'Planar component 2', ...),
            (                                              'Axial angle', ...),
            (                                             'Planar angle', ...),
            (                                                        'Y', ...),
            ...
            ('Diff from Median adjac Cell surface area at 1 frame prior', ...),
            (   'Diff from Median adjac Nuclear height at 1 frame prior', ...),
            (     'Diff 

In [36]:
feature2predict = 'At least one daughter differentiated'

Xcols = divisions.columns[divisions.columns.get_level_values(1) != 'Meta']
X = divisions[Xcols]

# X = divisions.xs('Measurement',axis=1,level=1).dropna(axis=1)
y = divisions[feature2predict,'Meta']

I = ~y.isnull()
X = X[I]
y = y[I].astype(bool)

# estimator = ensemble.RandomForestClassifier()

# selector = feature_selection.RFECV(estimator,step=1).fit(X,y)

## Recursive feature elimination CV: division

In [37]:
selector.get_feature_names_out()

NameError: name 'selector' is not defined

## RFECV: Frame prev to division